# Gemma detector

In [1]:
import accelerate
import torch
from datasets import load_dataset, concatenate_datasets
from torch import nn
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Gemma2ForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sentence_transformers import SentenceTransformer
from getpass import getpass
import numpy as np
import random

In [2]:
token = getpass()

 ········


In [3]:
import warnings
warnings.filterwarnings("ignore", message="Was asked to gather along dimension 0, but all input tensors were scalars")

## Prepare the model and data

### Detector architecture

In [4]:
model = SentenceTransformer("google/embeddinggemma-300m", token=token)

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

### In addition to the general embedding, we utilize context embedding separately to highlight the significance of context. To connect these two vectors, we employ a bottleneck layer and subsequently apply a nonlinear layer to extract features

In [5]:
class GemmaDetector(nn.Module):
    def __init__(self, embedding_dim, prj_dim):
        super().__init__()
        self.under_prj = nn.Linear(2 * embedding_dim, prj_dim)
        self.up_prj = nn.Linear(prj_dim, embedding_dim)
        self.prediction_layer = nn.Linear(embedding_dim, 2)
    def forward(self, encoded_context, encoded_sentences, labels):
        x = torch.hstack((encoded_context, encoded_sentences))
        x = self.under_prj(x)
        x = self.up_prj(x)
        x = nn.functional.relu(x)
        logits = self.prediction_layer(x)
        loss_func = nn.CrossEntropyLoss()
        loss = loss_func(logits, labels)
        return SequenceClassifierOutput(logits=logits, loss=loss)

### Data preprocessing

In [6]:
ds = load_dataset("pminervini/HaluEval", "qa", split="data", token=token)

In [7]:
def preprocess_right(examples):
    texts = []
    labels = []
    contexts = []
    for context, question, r_answer, h_answer in zip(examples["knowledge"], examples["question"], examples["right_answer"], examples["hallucinated_answer"]):
        texts.append(
            f"Is the following answer supported by the context and question? Context: '{context}', Question= '{question}', answer = '{r_answer}'"
        )
        contexts.append(context)
        labels.append(0)    
    return {'encoded_sentences': model.encode(texts), 'labels': torch.tensor(labels), 'encoded_context': model.encode(contexts)}

def preprocess_hals(examples):
    texts = []
    labels = []
    contexts = []
    for context, question, r_answer, h_answer in zip(examples["knowledge"], examples["question"], examples["right_answer"], examples["hallucinated_answer"]):
        texts.append(
            f"Is the following answer supported by the context and question? Context: '{context}', Question= '{question}', answer = '{h_answer}'"
        )
        contexts.append(context)
        labels.append(1)    
    return {'encoded_sentences': model.encode(texts), 'labels': torch.tensor(labels), 'encoded_context': model.encode(contexts)}

def preprocess_neg(examples):
    texts = []
    labels = []
    contexts = []
    for context, question, r_answer, h_answer in zip(examples["knowledge"], examples["question"], examples["right_answer"], examples["hallucinated_answer"]):
        texts.append(
            f"Is the following answer supported by the context and question? Context: '{context}', Question= '{question}', answer = '{r_answer}'"
        )
        contexts.append(context)
        labels.append(1)    
    return {'encoded_sentences': model.encode(texts), 'labels': torch.tensor(labels), 'encoded_context': model.encode(contexts)}



In [8]:
def preprocess_testset(examples):
    texts = []
    labels = []
    contexts = []
    for context, question, answer, halluc in zip(examples["knowledge"], examples["question"], examples["answer"], examples["hallucination"]):
        texts.append(
            f"Is the following answer supported by the context and question? Context: '{context}', Question= '{question}', answer = '{answer}'"
        )
        labels.append(1 if halluc == 'yes' else 0)
        contexts.append(context)
        
    return {'encoded_sentences': model.encode(texts), 'labels': torch.tensor(labels), 'encoded_context': model.encode(contexts)}

    


### Negative sampling(it is necessary here)

In [9]:
n_neg = int(len(ds) * 0.3)
indices = random.sample(range(len(ds)), n_neg)
ds_for_neg = ds.select(indices)
all_knowledges = ds_for_neg["knowledge"]
shuffled_knowledges = random.sample(all_knowledges, len(all_knowledges))
ds_neg = ds_for_neg.remove_columns("knowledge").add_column("knowledge", shuffled_knowledges)

Flattening the indices:   0%|          | 0/3000 [00:00<?, ? examples/s]

### Finally the train dataset

In [10]:
ds_emb_right = ds.map(preprocess_right, batched=True)
ds_emb_hal = ds.map(preprocess_hals, batched=True)
ds_emb_neg = ds_neg.map(preprocess_neg, batched=True)


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [11]:
combined_ds = concatenate_datasets([ds_emb_right, ds_emb_hal, ds_emb_neg]).shuffle(seed=42)

## Train

### This is enough for binary classification

In [12]:
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    
    return {
        "f1": f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall": recall_score(labels, preds),
        "accuracy": (preds == labels).mean()
    }


In [13]:
ds_test = load_dataset("pminervini/HaluEval", "qa_samples", split="data", token=token)

In [14]:
ds_test_emb = ds_test.map(preprocess_testset, batched=True)

In [15]:
detector = GemmaDetector(768, 300)

In [16]:
args = TrainingArguments(
    output_dir="/userspace/srm/gemma-hallucination-check",
    per_device_train_batch_size=64,
    num_train_epochs=7,
    logging_steps=100,
    learning_rate=0.01,
    eval_strategy="steps", 
    eval_steps=100,
    lr_scheduler_type='linear'
)
trainer = Trainer(model=detector, args=args, train_dataset=combined_ds, compute_metrics=compute_metrics, eval_dataset=ds_test_emb)

In [17]:
trainer.train()

Step,Training Loss,Validation Loss,F1,Precision,Recall,Accuracy
100,0.671464,0.594925,0.640573,0.728383,0.571657,0.678600
200,0.518855,0.461925,0.775912,0.796803,0.756088,0.781200
300,0.440674,0.434649,0.814340,0.772093,0.861477,0.803200
400,0.418966,0.395704,0.807588,0.883749,0.743513,0.822500
500,0.374333,0.363896,0.834244,0.863180,0.807186,0.839300
600,0.361094,0.360351,0.842276,0.856259,0.828743,0.844500
700,0.345096,0.334080,0.858486,0.860293,0.856687,0.858500
800,0.321323,0.330995,0.860978,0.845488,0.877046,0.858100
900,0.321892,0.309457,0.861529,0.903417,0.823353,0.867400
1000,0.295357,0.300899,0.869817,0.902381,0.839521,0.874100


TrainOutput(global_step=1260, training_loss=0.3809236541626945, metrics={'train_runtime': 185.4129, 'train_samples_per_second': 868.332, 'train_steps_per_second': 6.796, 'total_flos': 0.0, 'train_loss': 0.3809236541626945, 'epoch': 7.0})